In [1]:
#Importing the libraries 
import numpy as np
import pandas as pd 
import torch 
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.utils.data
from torch.autograd import Variable

In [2]:
#Importing the Dataset
movies = pd.read_csv('D:/Download/Part 5 - Boltzmann Machines (BM)/ml-1m/movies.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')
#movies
users = pd.read_csv('D:/Download/Part 5 - Boltzmann Machines (BM)/ml-1m/users.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')
#users
ratings = pd.read_csv('D:/Download/Part 5 - Boltzmann Machines (BM)/ml-1m/ratings.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')
#ratings

In [3]:
#Prepering the Training set and the Testing set
training_set = pd.read_csv('D:/Download/Part 5 - Boltzmann Machines (BM)/ml-100k/u1.base', delimiter = '\t')
#training_set
training_set = np.array(training_set, dtype ='int')
#training_set
#Prepering the Training set and the Testing set
test_set = pd.read_csv('D:/Download/Part 5 - Boltzmann Machines (BM)/ml-100k/u1.test', delimiter = '\t')
#test_set
test_set = np.array(test_set, dtype ='int')
#test_set


In [4]:
#Get the number of users and movies 
#O logos pou to kanoume me auth thn texniki einai se periptosi pou 8eloume na dwkimasoume
#diaforetika arxeia na briskei panta to megalitero kai se auta giati an baloume sigkekrimeno 
#ari8mo tote an to treksoume me alla arxeia den 8a einai swsto
nb_users = int(max(max(training_set[:,0]), max(test_set[:,0])))
nb_movies = int(max(max(training_set[:,1]), max(test_set[:,1])))
#print(nb_users)
#print(nb_movies)


In [5]:
#Comverting the data into an array with users in lines and movies in columns
def convert(data):
    #data = np.array(data, dtype='int')
    new_data = []
    for id_users in range(1,nb_users + 1):
        id_movies = data[:, 1][data[:,0] == id_users]
        id_ratings = data[:, 2][data[:,0] == id_users]
        ratings = np.zeros(nb_movies)
        ratings[id_movies - 1] = id_ratings
        new_data.append(list(ratings))
    return new_data

training_set = convert(training_set)
test_set = convert(test_set)
#print(training_set[0])
#print(test_set)

In [6]:
#Converting the data into Torch tensors
training_set = torch.FloatTensor(training_set)
test_set = torch.FloatTensor(test_set)
#print(training_set[0])
#print(test_set[0])


In [7]:
#Creating the architecture of the Neural Network
class SAE(nn.Module):
    def __init__(self, ):
        super(SAE, self).__init__()
        self.fc1 = nn.Linear(nb_movies, 20)
        self.fc2 = nn.Linear(20, 10)
        self.fc3 = nn.Linear(10, 20)
        self.fc4 = nn.Linear(20, nb_movies)
        self.activation = nn.Sigmoid()
    def forward(self, x):
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.activation(self.fc3(x))
        x = self.fc4(x)
        return x

sae = SAE()
criterion = nn.MSELoss()
optimizer = optim.RMSprop(sae.parameters(), lr = 0.01, weight_decay = 0.5)


In [14]:
#Training The SAE
nb_epoch = 200
for epoch in range(1, nb_epoch + 1):
    train_loss = 0
    s = 0.0
    for id_user in range(nb_users):
        input = Variable(training_set[id_user]).unsqueeze(0)
        target = input.clone()
        if torch.sum(target.data > 0) > 0 :
            output = sae(input)
            target.require_grad = False
            output[target == 0] = 0
            loss = criterion(output, target)
            mean_corrector = nb_movies/float(torch.sum(target.data > 0) + 1e-10) 
            loss.backward()
            train_loss += np.sqrt(loss.item() * mean_corrector)
            s += 1.0
            optimizer.step()
    print('epoch: '+ str(epoch) + ' loss: ' + str(train_loss/s))

epoch: 1 loss: 1.0964041984393031
epoch: 2 loss: 1.053338412501103
epoch: 3 loss: 1.0383938070737473
epoch: 4 loss: 1.0309030508589054
epoch: 5 loss: 1.0266835518993804
epoch: 6 loss: 1.0237800211656258
epoch: 7 loss: 1.0219885684505976
epoch: 8 loss: 1.0207017499752726
epoch: 9 loss: 1.0196799474239193
epoch: 10 loss: 1.0185848631327348
epoch: 11 loss: 1.0185493736944724
epoch: 12 loss: 1.0178513275958911
epoch: 13 loss: 1.0174292135834138
epoch: 14 loss: 1.0170689536829232
epoch: 15 loss: 1.017048758322035
epoch: 16 loss: 1.0167631648507993
epoch: 17 loss: 1.0164608028835986
epoch: 18 loss: 1.0165327721391595
epoch: 19 loss: 1.0163959043697577
epoch: 20 loss: 1.0159524313539219
epoch: 21 loss: 1.0159168684814226
epoch: 22 loss: 1.015891358695707
epoch: 23 loss: 1.015789076436456
epoch: 24 loss: 1.0158282492079382
epoch: 25 loss: 1.0156806590569512
epoch: 26 loss: 1.0152522061921507
epoch: 27 loss: 1.0151067523655581
epoch: 28 loss: 1.0131383471802309
epoch: 29 loss: 1.01144881222778


In [18]:
test_loss = 0
s = 0.0
for id_user in range(nb_users):
        input = Variable(training_set[id_user]).unsqueeze(0)
        target = Variable(test_set[id_user])
        if torch.sum(target.data > 0) > 0 :
            output = sae(input)
            target.require_grad = False
            output[0,target == 0] = 0
            loss = criterion(output, target)
            mean_corrector = nb_movies/float(torch.sum(target.data > 0) + 1e-10) 
            test_loss += np.sqrt(loss.item() * mean_corrector)
            s += 1.0
print('loss: ' + str(test_loss/s))

D:\Anacoda\lib\site-packages\torch\nn\modules\loss.py:530: UserWarning: Using a target size (torch.Size([1682])) that is different to the input size (torch.Size([1, 1682])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


loss: 0.9474387059763989
